In [ ]:
from unsloth import FastLanguageModel
import time

---

In [ ]:
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

In [ ]:
# adapter_path = "outputs/run_20250505_203329/checkpoint-2300"
# adapter_path = "outputs/run_20250505_231545/checkpoint-2300"
# adapter_path = "outputs/run_20250506_000331/checkpoint-3500"
# adapter_path = "outputs/run_20250506_103144/checkpoint-3708"
adapter_path = "outputs/run_20250506_194631/checkpoint-2300"

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=adapter_path,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

In [ ]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
model.eval() # Set the model to evaluation mode

---

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

In [ ]:
# instruction = "\n".join(conversation[max(-10, -len(conversation)):])
# instruction = "\n".join(conversation[-5:])
instruction = "Какой город - северная столица России?"
input_context = "" # Provide input context if your task requires it
formatted_prompt = alpaca_prompt.format(instruction, input_context, "") # Leave response blank

inputs = tokenizer(
    [formatted_prompt],
    return_tensors = "pt"
).to("cuda")

In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens = 128,
    use_cache = True,
    eos_token_id = tokenizer.eos_token_id,
    pad_token_id = tokenizer.pad_token_id,
    # Add other generation parameters like top_p, temperature if needed
    top_p=0.95,
    temperature=0.9,
)

In [ ]:
decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

# Extract only the response part after "### Response:"
response_start_tag = "### Response:"
response_index = decoded_output.find(response_start_tag)
if response_index != -1:
    response_text = decoded_output[response_index + len(response_start_tag):].strip()
    # Remove the EOS token if it's present at the end
    if response_text.endswith(EOS_TOKEN):
        response_text = response_text[:-len(EOS_TOKEN)].strip()
else:
    # Fallback if the tag isn't found (might indicate an issue)
    print("Warning: '### Response:' tag not found in output.")
    response_text = decoded_output # Or handle as appropriate

print("Instruction:", instruction)
print()
print("Леша:", response_text)


---

In [ ]:

conversation = [
    "Соня: Доброе утро!🌞",
    "Соня: Как дела?",
]

print("\n".join(conversation))
for i in range(10):
    instruction = "\n".join(conversation)
    input_context = "" # Provide input context if your task requires it
    formatted_prompt = alpaca_prompt.format(instruction, input_context, "") # Leave response blank

    inputs = tokenizer(
        [formatted_prompt],
        return_tensors = "pt"
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens = 64,
        use_cache = True,
        eos_token_id = tokenizer.eos_token_id,
        pad_token_id = tokenizer.pad_token_id,
        # Add other generation parameters like top_p, temperature if needed
        # top_p=0.9,
        temperature=0.7,
    )

    decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    # Extract only the response part after "### Response:"
    response_start_tag = "### Response:"
    response_index = decoded_output.find(response_start_tag)
    if response_index != -1:
        response_text = decoded_output[response_index + len(response_start_tag):].strip()
        # Remove the EOS token if it's present at the end
        if response_text.endswith(EOS_TOKEN):
            response_text = response_text[:-len(EOS_TOKEN)].strip()
    else:
        # Fallback if the tag isn't found (might indicate an issue)
        print("Warning: '### Response:' tag not found in output.")
        response_text = decoded_output # Or handle as appropriate

    # print("Instruction:", instruction)
    # print()
    # print("Леша:", response_text)

    conversation.append(f"Леша: {response_text}")
    print(conversation[-1])

    time.sleep(1)

    input_context = input("Соня: ")
    if input_context == "exit":
        break
    conversation.append(f"Соня: {input_context}")
    print(conversation[-1])

---

In [ ]:
custom_prompt = """Ты ассистент, который учавствует в диалоге между Лешей и Соней. Ниже приведено сообщение, на которое вы должны ответить максимально похоже на то, как бы это сделал Леша

### Сообщение:
{}

### Ответ:
{}"""

EOS_TOKEN = tokenizer.eos_token

# ================================

input = "Как твои дела?"

# ================================

formatted_prompt = custom_prompt.format(input, "") # Leave response blank

inputs = tokenizer(
    [formatted_prompt],
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens = 1024,
    use_cache = True,
    eos_token_id = tokenizer.eos_token_id,
    pad_token_id = tokenizer.pad_token_id,
    top_p=0.9,
    temperature=0.7,
    repetition_penalty=1.1,
    do_sample=True,
)

decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

# Extract only the response part after "### Response:"
response_start_tag = "### Ответ:"
response_index = decoded_output.find(response_start_tag)
if response_index != -1:
    response_text = decoded_output[response_index + len(response_start_tag):].strip()
    # Remove the EOS token if it's present at the end
    if response_text.endswith(EOS_TOKEN):
        response_text = response_text[:-len(EOS_TOKEN)].strip()
else:
    # Fallback if the tag isn't found (might indicate an issue)
    print("Warning: '### Ответ:' tag not found in output.")
    response_text = decoded_output # Or handle as appropriate

print("Input:", input)
print()
print("Леша:", response_text)


---

In [ ]:
custom_context_prompt = """Ты ассистент, который учавствует в диалоге между Лешей и Соней. Прими во внимание контекст диалога. Твоя задача ответить на приведенное ниже сообщение максимально похоже на то, как бы это сделал Леша

### Контекст:
{}

### Сообщение:
{}

### Ответ:
{}"""

EOS_TOKEN = tokenizer.eos_token

# ================================

context = [
]
input = "Как ты думаешь, чем смысл жизни?"

# ================================

formatted_prompt = custom_context_prompt.format("/n".join(context), input, "") # Leave response blank

inputs = tokenizer(
    [formatted_prompt],
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens = 2048,
    use_cache = True,
    eos_token_id = tokenizer.eos_token_id,
    pad_token_id = tokenizer.pad_token_id,
    top_p=0.9,
    temperature=0.7,
    repetition_penalty=1.2,
    do_sample=True,
)

decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

# Extract only the response part after "### Response:"
response_start_tag = "### Ответ:"
response_index = decoded_output.find(response_start_tag)
if response_index != -1:
    response_text = decoded_output[response_index + len(response_start_tag):].strip()
    # Remove the EOS token if it's present at the end
    if response_text.endswith(EOS_TOKEN):
        response_text = response_text[:-len(EOS_TOKEN)].strip()
else:
    # Fallback if the tag isn't found (might indicate an issue)
    print("Warning: '### Ответ:' tag not found in output.")
    response_text = decoded_output # Or handle as appropriate

print("\n".join(context) + "\nСоня: " + input)
print()
print("🤖:", response_text)


---

In [ ]:
import json
import random

In [ ]:
with open("dataset_alpaca_repl.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
subdata = random.sample(data, 10)

In [ ]:
for sub in subdata:
    instruction = sub['instruction']
    input_context = "" # Provide input context if your task requires it
    formatted_prompt = custom_prompt.format(instruction, "") # Leave response blank

    inputs = tokenizer(
        [formatted_prompt],
        return_tensors = "pt"
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens = 128,
        use_cache = True,
        eos_token_id = tokenizer.eos_token_id,
        pad_token_id = tokenizer.pad_token_id,
        # Add other generation parameters like top_p, temperature if needed
        top_p=0.95,
        temperature=0.7,
        repetition_penalty=1.2,
        do_sample=True,
    )

    decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    # Extract only the response part after "### Response:"
    response_start_tag = "### Ответ:"
    response_index = decoded_output.find(response_start_tag)
    if response_index != -1:
        response_text = decoded_output[response_index + len(response_start_tag):].strip()
        # Remove the EOS token if it's present at the end
        if response_text.endswith(EOS_TOKEN):
            response_text = response_text[:-len(EOS_TOKEN)].strip()
    else:
        # Fallback if the tag isn't found (might indicate an issue)
        print("Warning: '### Ответ:' tag not found in output.")
        response_text = decoded_output # Or handle as appropriate

    print("Instruction:", instruction)
    print("output:", sub['output'])
    print("Леша:", response_text)
    print()